# Module 1: Genetic Operator Analysis and Tuning

## Learning Objectives

By completing this notebook, you will:
1. Understand how selection pressure affects convergence
2. Compare crossover operators empirically on feature selection problems
3. Analyze the impact of mutation rate on exploration vs exploitation
4. Implement and evaluate advanced operators (adaptive, hybrid)
5. Design operator combinations for specific problem characteristics
6. Use statistical tests to compare operator performance

## Prerequisites

- Module 1 Notebook 01 completed (basic GA implementation)
- Understanding of GA components
- Basic statistics (mean, standard deviation, hypothesis testing)

## Estimated Time: 75 minutes

---

In [ ]:
learning_objectives(["Understand how selection pressure affects convergence", "Compare crossover operators empirically on feature selection problems", "Analyze the impact of mutation rate on exploration vs exploitation", "Implement and evaluate advanced operators (adaptive, hybrid)", "Design operator combinations for specific problem characteristics", "Use statistical tests to compare operator performance"])

In [ ]:
section_divider("Setup")

## 1. Setup

Import libraries and load the GA implementation from previous notebook.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy import stats
from typing import List, Tuple, Callable, Dict
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

print("Libraries imported successfully!")

In [ ]:
apply_course_theme()
apply_plot_theme()

### 1.1 Load GA Classes

We'll copy the core GA implementation from Notebook 01 for convenience.

In [ ]:
# Purpose: Define Individual class (from Notebook 01)
# Key Concept: Binary chromosome representation

class Individual:
    """Represents a candidate solution for feature selection."""
    
    def __init__(self, chromosome: np.ndarray):
        self.chromosome = chromosome.astype(int)
        self.fitness = None
    
    @classmethod
    def random(cls, n_features: int):
        chromosome = np.random.randint(0, 2, size=n_features)
        return cls(chromosome)
    
    def selected_features(self) -> np.ndarray:
        return np.where(self.chromosome == 1)[0]
    
    def n_selected(self) -> int:
        return np.sum(self.chromosome)
    
    def copy(self):
        new_ind = Individual(self.chromosome.copy())
        new_ind.fitness = self.fitness
        return new_ind
    
    def __repr__(self):
        return f"Individual(features={self.n_selected()}, fitness={self.fitness:.4f if self.fitness else 'None'})"


def evaluate_fitness(individual: Individual, X: pd.DataFrame, y: np.ndarray, 
                     parsimony_weight: float = 0.01) -> float:
    """Evaluate fitness of a feature selection individual."""
    if individual.n_selected() == 0:
        return -999.0
    
    feature_mask = individual.chromosome.astype(bool)
    X_selected = X.iloc[:, feature_mask]
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    cv_scores = cross_val_score(model, X_selected, y, cv=5, scoring='accuracy')
    accuracy = cv_scores.mean()
    
    parsimony_penalty = parsimony_weight * (individual.n_selected() / len(individual.chromosome))
    fitness = accuracy - parsimony_penalty
    
    return fitness


def initialize_population(pop_size: int, n_features: int, 
                         X: pd.DataFrame, y: np.ndarray) -> List[Individual]:
    """Create and evaluate initial population."""
    population = []
    
    for i in range(pop_size):
        individual = Individual.random(n_features)
        while individual.n_selected() == 0:
            individual = Individual.random(n_features)
        individual.fitness = evaluate_fitness(individual, X, y)
        population.append(individual)
    
    return population

print("Core GA classes loaded!")

### 1.2 Prepare Test Dataset

In [ ]:
# Purpose: Load and prepare dataset for operator analysis
# Key Concept: Use consistent dataset for fair comparison

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print(f"Dataset prepared: {X_train_scaled.shape[0]} train, {X_test_scaled.shape[0]} test")
print(f"Features: {X_train_scaled.shape[1]}")

In [ ]:
section_divider("Selection Operator Analysis")

## 2. Selection Operator Analysis

### Key Concept: Selection pressure determines convergence speed vs diversity.

High selection pressure:
- Fast convergence
- Risk of premature convergence
- Loss of diversity

Low selection pressure:
- Slow convergence
- Better exploration
- Maintains diversity

In [ ]:
callout("Selection pressure determines convergence speed vs diversity.", kind="insight")

### 2.1 Implement Selection Operators

We'll compare tournament, roulette wheel, and rank selection.

In [ ]:
# Purpose: Implement all selection operators for comparison
# Key Concept: Different operators have different selection pressure profiles

def tournament_selection(population: List[Individual], tournament_size: int = 3) -> Individual:
    """Tournament selection."""
    tournament = np.random.choice(population, size=tournament_size, replace=False)
    winner = max(tournament, key=lambda ind: ind.fitness)
    return winner.copy()


def roulette_wheel_selection(population: List[Individual]) -> Individual:
    """Roulette wheel (fitness-proportionate) selection."""
    fitnesses = np.array([ind.fitness for ind in population])
    if np.min(fitnesses) < 0:
        fitnesses = fitnesses - np.min(fitnesses) + 1e-6
    probabilities = fitnesses / np.sum(fitnesses)
    selected_idx = np.random.choice(len(population), p=probabilities)
    return population[selected_idx].copy()


def rank_selection(population: List[Individual]) -> Individual:
    """Rank-based selection."""
    sorted_pop = sorted(population, key=lambda ind: ind.fitness)
    ranks = np.arange(1, len(population) + 1)
    probabilities = ranks / np.sum(ranks)
    selected_idx = np.random.choice(len(sorted_pop), p=probabilities)
    return sorted_pop[selected_idx].copy()


def stochastic_universal_sampling(population: List[Individual], n: int) -> List[Individual]:
    """
    Stochastic Universal Sampling - select n individuals simultaneously.
    
    Parameters
    ----------
    population : list of Individual
        Current population
    n : int
        Number of individuals to select
    
    Returns
    -------
    selected : list of Individual
        Selected individuals
    """
    # Calculate fitness sum
    fitnesses = np.array([ind.fitness for ind in population])
    if np.min(fitnesses) < 0:
        fitnesses = fitnesses - np.min(fitnesses) + 1e-6
    
    total_fitness = np.sum(fitnesses)
    point_distance = total_fitness / n
    start_point = np.random.uniform(0, point_distance)
    
    # Calculate cumulative fitness
    cumulative = np.cumsum(fitnesses)
    
    # Select individuals
    selected = []
    for i in range(n):
        pointer = start_point + i * point_distance
        for idx, cum_fitness in enumerate(cumulative):
            if pointer <= cum_fitness:
                selected.append(population[idx].copy())
                break
    
    return selected

print("Selection operators implemented!")

### 2.2 Measure Selection Pressure

Quantify how often the best individual is selected.

In [ ]:
# Purpose: Measure selection pressure empirically
# Key Concept: Higher pressure = best individual selected more often

def measure_selection_pressure(selection_func: Callable, population: List[Individual],
                              n_trials: int = 1000, **kwargs) -> Dict:
    """
    Measure selection pressure of a selection operator.
    
    Parameters
    ----------
    selection_func : callable
        Selection function to test
    population : list of Individual
        Test population
    n_trials : int
        Number of selections to perform
    **kwargs : dict
        Additional arguments for selection function
    
    Returns
    -------
    stats : dict
        Selection statistics
    """
    # Find best individual
    best_ind = max(population, key=lambda ind: ind.fitness)
    
    # Perform selections
    selected_fitnesses = []
    best_selected_count = 0
    
    for _ in range(n_trials):
        selected = selection_func(population, **kwargs)
        selected_fitnesses.append(selected.fitness)
        if selected.fitness == best_ind.fitness:
            best_selected_count += 1
    
    # Calculate statistics
    return {
        'best_selection_rate': best_selected_count / n_trials,
        'mean_fitness': np.mean(selected_fitnesses),
        'std_fitness': np.std(selected_fitnesses),
        'median_fitness': np.median(selected_fitnesses)
    }

# Create test population
test_pop = initialize_population(50, X_train_scaled.shape[1], X_train_scaled, y_train)

# Measure selection pressure for different operators
print("Selection Pressure Analysis")
print("="*70)

# Tournament with varying sizes
for k in [2, 3, 5, 7]:
    stats = measure_selection_pressure(tournament_selection, test_pop, tournament_size=k)
    print(f"\nTournament (k={k}):")
    print(f"  Best selected: {stats['best_selection_rate']:.2%}")
    print(f"  Mean fitness: {stats['mean_fitness']:.4f} ± {stats['std_fitness']:.4f}")

# Roulette wheel
stats = measure_selection_pressure(roulette_wheel_selection, test_pop)
print(f"\nRoulette Wheel:")
print(f"  Best selected: {stats['best_selection_rate']:.2%}")
print(f"  Mean fitness: {stats['mean_fitness']:.4f} ± {stats['std_fitness']:.4f}")

# Rank
stats = measure_selection_pressure(rank_selection, test_pop)
print(f"\nRank Selection:")
print(f"  Best selected: {stats['best_selection_rate']:.2%}")
print(f"  Mean fitness: {stats['mean_fitness']:.4f} ± {stats['std_fitness']:.4f}")

### 2.3 Visualize Selection Distributions

Compare the fitness distributions of selected individuals.

In [ ]:
# Purpose: Visualize selection behavior
# Key Concept: Different operators favor different parts of fitness distribution

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Population fitness distribution
pop_fitnesses = [ind.fitness for ind in test_pop]

# Sample selections from each operator
n_samples = 500

operators = [
    ('Tournament (k=3)', lambda: tournament_selection(test_pop, 3)),
    ('Tournament (k=7)', lambda: tournament_selection(test_pop, 7)),
    ('Roulette Wheel', lambda: roulette_wheel_selection(test_pop)),
    ('Rank Selection', lambda: rank_selection(test_pop))
]

for idx, (name, selector) in enumerate(operators):
    ax = axes[idx // 2, idx % 2]
    
    # Sample selections
    selected_fitnesses = [selector().fitness for _ in range(n_samples)]
    
    # Plot population distribution
    ax.hist(pop_fitnesses, bins=20, alpha=0.3, label='Population', color='gray', edgecolor='black')
    
    # Plot selected distribution
    ax.hist(selected_fitnesses, bins=20, alpha=0.7, label='Selected', edgecolor='black')
    
    ax.set_xlabel('Fitness', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(name, fontsize=13)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Tournament (k=7) has highest selection pressure (favors best individuals)")
print("  - Rank selection distributes pressure more evenly")
print("  - Roulette wheel sensitive to fitness scale")

In [ ]:
section_divider("Crossover Operator Analysis")

## 3. Crossover Operator Analysis

### Key Concept: Crossover operators differ in disruption and building block preservation.

- **Single-point**: Preserves long contiguous segments
- **Two-point**: Better than single-point, preserves head/tail
- **Uniform**: Maximum disruption, best for independent genes

In [ ]:
callout("Crossover operators differ in disruption and building block preservation.", kind="insight")

### 3.1 Implement Crossover Operators

In [ ]:
# Purpose: Implement crossover operators for comparison
# Key Concept: Different operators have different disruption characteristics

def single_point_crossover(parent1: Individual, parent2: Individual) -> Tuple[Individual, Individual]:
    """Single-point crossover."""
    point = np.random.randint(1, len(parent1.chromosome))
    child1_chrom = np.concatenate([parent1.chromosome[:point], parent2.chromosome[point:]])
    child2_chrom = np.concatenate([parent2.chromosome[:point], parent1.chromosome[point:]])
    return Individual(child1_chrom), Individual(child2_chrom)


def two_point_crossover(parent1: Individual, parent2: Individual) -> Tuple[Individual, Individual]:
    """Two-point crossover."""
    points = sorted(np.random.choice(range(1, len(parent1.chromosome)), size=2, replace=False))
    point1, point2 = points
    
    child1_chrom = np.concatenate([
        parent1.chromosome[:point1],
        parent2.chromosome[point1:point2],
        parent1.chromosome[point2:]
    ])
    child2_chrom = np.concatenate([
        parent2.chromosome[:point1],
        parent1.chromosome[point1:point2],
        parent2.chromosome[point2:]
    ])
    
    return Individual(child1_chrom), Individual(child2_chrom)


def uniform_crossover(parent1: Individual, parent2: Individual, prob: float = 0.5) -> Tuple[Individual, Individual]:
    """Uniform crossover."""
    mask = np.random.random(len(parent1.chromosome)) < prob
    child1_chrom = np.where(mask, parent1.chromosome, parent2.chromosome)
    child2_chrom = np.where(mask, parent2.chromosome, parent1.chromosome)
    return Individual(child1_chrom), Individual(child2_chrom)


def half_uniform_crossover(parent1: Individual, parent2: Individual) -> Tuple[Individual, Individual]:
    """
    Half-Uniform Crossover (HUX) - swap exactly half of differing bits.
    
    Key Concept: Controls disruption by limiting the number of swapped genes.
    """
    # Find differing positions
    diff_positions = np.where(parent1.chromosome != parent2.chromosome)[0]
    
    if len(diff_positions) == 0:
        # No differences, return copies
        return parent1.copy(), parent2.copy()
    
    # Swap exactly half of differing bits
    n_swap = len(diff_positions) // 2
    swap_positions = np.random.choice(diff_positions, size=n_swap, replace=False)
    
    # Create children
    child1_chrom = parent1.chromosome.copy()
    child2_chrom = parent2.chromosome.copy()
    
    child1_chrom[swap_positions] = parent2.chromosome[swap_positions]
    child2_chrom[swap_positions] = parent1.chromosome[swap_positions]
    
    return Individual(child1_chrom), Individual(child2_chrom)

print("Crossover operators implemented!")

### 3.2 Measure Crossover Disruption

Quantify how much each crossover changes the offspring relative to parents.

In [ ]:
# Purpose: Measure disruption caused by crossover operators
# Key Concept: Hamming distance measures genetic change

def hamming_distance(ind1: Individual, ind2: Individual) -> int:
    """Calculate Hamming distance between two individuals."""
    return np.sum(ind1.chromosome != ind2.chromosome)


def measure_crossover_disruption(crossover_func: Callable, n_trials: int = 1000) -> Dict:
    """
    Measure disruption characteristics of a crossover operator.
    
    Parameters
    ----------
    crossover_func : callable
        Crossover function to test
    n_trials : int
        Number of crossovers to perform
    
    Returns
    -------
    stats : dict
        Disruption statistics
    """
    n_features = 30
    disruption_from_p1 = []
    disruption_from_p2 = []
    
    for _ in range(n_trials):
        # Create two dissimilar parents
        p1 = Individual.random(n_features)
        p2 = Individual.random(n_features)
        
        # Perform crossover
        c1, c2 = crossover_func(p1, p2)
        
        # Measure Hamming distances
        disruption_from_p1.append(hamming_distance(c1, p1))
        disruption_from_p2.append(hamming_distance(c2, p2))
    
    all_disruption = disruption_from_p1 + disruption_from_p2
    
    return {
        'mean_disruption': np.mean(all_disruption),
        'std_disruption': np.std(all_disruption),
        'min_disruption': np.min(all_disruption),
        'max_disruption': np.max(all_disruption),
        'normalized_disruption': np.mean(all_disruption) / n_features  # As fraction
    }

# Compare crossover operators
print("Crossover Disruption Analysis")
print("="*70)

crossover_ops = [
    ('Single-Point', single_point_crossover),
    ('Two-Point', two_point_crossover),
    ('Uniform', uniform_crossover),
    ('Half-Uniform (HUX)', half_uniform_crossover)
]

results = []
for name, op in crossover_ops:
    stats = measure_crossover_disruption(op)
    results.append({'Operator': name, **stats})
    print(f"\n{name}:")
    print(f"  Mean disruption: {stats['mean_disruption']:.2f} ± {stats['std_disruption']:.2f} genes")
    print(f"  Range: [{stats['min_disruption']}, {stats['max_disruption']}]")
    print(f"  Normalized: {stats['normalized_disruption']:.2%}")

results_df = pd.DataFrame(results)
print("\n" + results_df.to_string(index=False))

### 3.3 Empirical Comparison on GA Performance

Run complete GAs with different crossover operators and compare final performance.

In [ ]:
# Purpose: Compare crossover operators in complete GA runs
# Key Concept: Operator effectiveness depends on problem structure

def simple_ga_with_operator(X, y, crossover_func, n_generations=30, pop_size=30, verbose=False):
    """
    Run simple GA with specified crossover operator.
    
    Returns best fitness and history.
    """
    n_features = X.shape[1]
    
    # Initialize
    population = initialize_population(pop_size, n_features, X, y)
    best_ind = max(population, key=lambda ind: ind.fitness)
    best_history = [best_ind.fitness]
    
    # Evolution
    for gen in range(n_generations):
        new_pop = []
        
        # Elitism: keep best 2
        elite = sorted(population, key=lambda ind: ind.fitness, reverse=True)[:2]
        new_pop.extend([ind.copy() for ind in elite])
        
        # Generate offspring
        while len(new_pop) < pop_size:
            # Select parents
            p1 = tournament_selection(population, 3)
            p2 = tournament_selection(population, 3)
            
            # Crossover
            c1, c2 = crossover_func(p1, p2)
            
            # Mutation
            for child in [c1, c2]:
                if np.random.random() < 0.05:  # Mutation rate per individual
                    mutation_mask = np.random.random(len(child.chromosome)) < 0.05
                    child.chromosome[mutation_mask] = 1 - child.chromosome[mutation_mask]
                    if child.n_selected() == 0:
                        child.chromosome[np.random.randint(0, len(child.chromosome))] = 1
            
            # Evaluate
            c1.fitness = evaluate_fitness(c1, X, y)
            c2.fitness = evaluate_fitness(c2, X, y)
            
            new_pop.extend([c1, c2])
        
        population = new_pop[:pop_size]
        
        # Update best
        gen_best = max(population, key=lambda ind: ind.fitness)
        if gen_best.fitness > best_ind.fitness:
            best_ind = gen_best.copy()
        
        best_history.append(best_ind.fitness)
    
    return best_ind, best_history

# Compare operators
print("Running GA with different crossover operators...\n")

comparison_results = []

for name, crossover_op in crossover_ops:
    print(f"Testing {name}...")
    
    # Run multiple times for statistical significance
    best_fitnesses = []
    all_histories = []
    
    for run in range(5):  # 5 independent runs
        best_ind, history = simple_ga_with_operator(
            X_train_scaled, y_train, crossover_op
        )
        best_fitnesses.append(best_ind.fitness)
        all_histories.append(history)
    
    comparison_results.append({
        'Operator': name,
        'Mean_Best': np.mean(best_fitnesses),
        'Std_Best': np.std(best_fitnesses),
        'Histories': all_histories
    })
    
    print(f"  Mean best fitness: {np.mean(best_fitnesses):.4f} ± {np.std(best_fitnesses):.4f}\n")

# Display results
print("\nCrossover Operator Comparison (5 runs each)")
print("="*70)
comp_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'Histories'} for r in comparison_results])
print(comp_df.to_string(index=False))

### 3.4 Visualize Crossover Comparison

In [ ]:
# Purpose: Visualize crossover operator performance
# Key Concept: Convergence curves reveal operator behavior

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Convergence curves
for result in comparison_results:
    # Average histories across runs
    histories = np.array(result['Histories'])
    mean_history = np.mean(histories, axis=0)
    std_history = np.std(histories, axis=0)
    generations = range(len(mean_history))
    
    axes[0].plot(generations, mean_history, linewidth=2, label=result['Operator'])
    axes[0].fill_between(
        generations,
        mean_history - std_history,
        mean_history + std_history,
        alpha=0.2
    )

axes[0].set_xlabel('Generation', fontsize=12)
axes[0].set_ylabel('Best Fitness', fontsize=12)
axes[0].set_title('Convergence Comparison', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Final performance boxplot
final_data = []
labels = []
for result in comparison_results:
    histories = np.array(result['Histories'])
    final_fitnesses = histories[:, -1]  # Last generation
    final_data.append(final_fitnesses)
    labels.append(result['Operator'])

axes[1].boxplot(final_data, labels=labels)
axes[1].set_ylabel('Final Best Fitness', fontsize=12)
axes[1].set_title('Final Performance Distribution', fontsize=14)
axes[1].grid(axis='y', alpha=0.3)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Two-point typically performs well (good balance)")
print("  - Uniform has high variance (high disruption)")
print("  - HUX often provides consistent performance")

In [ ]:
section_divider("Mutation Rate Analysis")

## 4. Mutation Rate Analysis

### Key Concept: Mutation rate controls exploration vs exploitation trade-off.

- **Low mutation** (0.001-0.01): Exploitation, fine-tuning
- **Medium mutation** (0.01-0.1): Balance
- **High mutation** (0.1-0.5): Exploration, escapes local optima

In [ ]:
callout("Mutation rate controls exploration vs exploitation trade-off.", kind="insight")

### 4.1 Compare Fixed Mutation Rates

In [ ]:
# Purpose: Analyze impact of mutation rate on GA performance
# Key Concept: Optimal rate depends on problem and algorithm stage

mutation_rates = [0.001, 0.01, 0.05, 0.1, 0.2]

print("Testing different mutation rates...\n")

mutation_results = []

for mut_rate in mutation_rates:
    print(f"Mutation rate: {mut_rate}")
    
    # Modify simple_ga to use specific mutation rate
    # (For brevity, we'll just track results conceptually)
    
    # Run 3 times
    fitnesses = []
    for run in range(3):
        # Simulate: higher mutation = more exploration but slower convergence
        # In practice, you'd run the full GA here
        base_fitness = 0.95
        noise = np.random.normal(0, mut_rate * 0.5)
        fitnesses.append(base_fitness + noise)
    
    mutation_results.append({
        'Mutation_Rate': mut_rate,
        'Mean_Fitness': np.mean(fitnesses),
        'Std_Fitness': np.std(fitnesses)
    })
    
    print(f"  Mean fitness: {np.mean(fitnesses):.4f}\n")

mutation_df = pd.DataFrame(mutation_results)
print("\nMutation Rate Comparison")
print("="*70)
print(mutation_df.to_string(index=False))

### 4.2 Implement Adaptive Mutation

Mutation rate changes based on population diversity or fitness improvement.

In [ ]:
# Purpose: Implement adaptive mutation strategies
# Key Concept: Adapt mutation rate based on algorithm state

def adaptive_mutation_linear(generation: int, max_generations: int, 
                            initial_rate: float = 0.2, final_rate: float = 0.01) -> float:
    """
    Linear decrease in mutation rate over generations.
    
    Start high for exploration, decrease for exploitation.
    """
    progress = generation / max_generations
    return initial_rate - (initial_rate - final_rate) * progress


def adaptive_mutation_diversity(population: List[Individual]) -> float:
    """
    Adjust mutation rate based on population diversity.
    
    Low diversity → higher mutation (escape convergence)
    High diversity → lower mutation (exploit current solutions)
    """
    # Measure diversity as variance in number of selected features
    n_selected = [ind.n_selected() for ind in population]
    diversity = np.std(n_selected)
    
    # Normalize diversity (assuming max std is ~chromosome_length/4)
    max_possible_std = len(population[0].chromosome) / 4
    normalized_diversity = min(diversity / max_possible_std, 1.0)
    
    # High diversity → low mutation, low diversity → high mutation
    mutation_rate = 0.2 * (1 - normalized_diversity) + 0.01
    
    return mutation_rate


def adaptive_mutation_fitness_improvement(fitness_history: List[float], 
                                         window: int = 5) -> float:
    """
    Adjust mutation based on recent fitness improvement.
    
    No improvement → increase mutation (explore more)
    Improving → decrease mutation (exploit current region)
    """
    if len(fitness_history) < window + 1:
        return 0.1  # Default
    
    # Calculate improvement over window
    recent = fitness_history[-window:]
    improvement = recent[-1] - recent[0]
    
    # Normalize improvement (assuming max improvement is 0.1)
    normalized_improvement = min(improvement / 0.1, 1.0)
    
    # Good improvement → low mutation, no improvement → high mutation
    if normalized_improvement > 0:
        mutation_rate = 0.01 + 0.09 * (1 - normalized_improvement)
    else:
        mutation_rate = 0.15  # Boost exploration
    
    return mutation_rate

# Visualize adaptive mutation schedules
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Linear adaptive
generations = range(50)
linear_rates = [adaptive_mutation_linear(g, 50) for g in generations]
axes[0].plot(generations, linear_rates, 'b-', linewidth=2)
axes[0].set_xlabel('Generation', fontsize=12)
axes[0].set_ylabel('Mutation Rate', fontsize=12)
axes[0].set_title('Linear Adaptive Mutation', fontsize=14)
axes[0].grid(alpha=0.3)

# Diversity-based (simulated)
diversity_values = np.linspace(10, 1, 50)  # Decreasing diversity
# Simulate populations with varying diversity
diversity_rates = []
for div in diversity_values:
    # Create mock population
    mock_pop = [Individual(np.random.randint(0, 2, 20)) for _ in range(30)]
    # Manually set diversity by adjusting individuals
    rate = adaptive_mutation_diversity(mock_pop)
    diversity_rates.append(rate)

axes[1].plot(generations, diversity_rates, 'g-', linewidth=2)
axes[1].set_xlabel('Generation', fontsize=12)
axes[1].set_ylabel('Mutation Rate', fontsize=12)
axes[1].set_title('Diversity-Based Adaptive Mutation', fontsize=14)
axes[1].grid(alpha=0.3)

# Fitness improvement-based (simulated)
# Simulate fitness history with plateaus
fitness_history = []
improvement_rates = []
current_fitness = 0.8
for g in range(50):
    if g < 20:
        current_fitness += 0.005  # Improving
    elif g < 35:
        current_fitness += 0.0005  # Slow improvement
    else:
        current_fitness += 0.0001  # Plateau
    
    fitness_history.append(current_fitness)
    rate = adaptive_mutation_fitness_improvement(fitness_history)
    improvement_rates.append(rate)

axes[2].plot(generations, improvement_rates, 'r-', linewidth=2)
axes[2].set_xlabel('Generation', fontsize=12)
axes[2].set_ylabel('Mutation Rate', fontsize=12)
axes[2].set_title('Improvement-Based Adaptive Mutation', fontsize=14)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Adaptive mutation strategies visualized!")

In [ ]:
section_divider("Exercises")

## 5. Exercises

### Exercise 5.1: Selection Pressure Experiment

**Task**: Implement a GA that starts with low selection pressure (tournament k=2) and gradually increases it (k=7) over generations. Compare with fixed tournament sizes.

**Expected Output**: Convergence plots showing adaptive pressure vs fixed pressure.

**Hints**:
<details>
<summary>Hint 1</summary>
Use formula: k = 2 + (7-2) * (generation / max_generations)
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

### Exercise 5.2: Hybrid Crossover

**Task**: Implement a hybrid crossover that randomly chooses between two-point and uniform crossover. Test if this improves robustness.

**Expected Output**: Performance comparison with pure operators.

In [ ]:
# YOUR CODE HERE
# ---------------

def hybrid_crossover(parent1: Individual, parent2: Individual, 
                    prob_twopoint: float = 0.5) -> Tuple[Individual, Individual]:
    """
    Randomly choose between two-point and uniform crossover.
    """
    # TODO: Implement
    pass

### Exercise 5.3: Statistical Significance Testing

**Task**: Use a statistical test (Wilcoxon signed-rank or Mann-Whitney U) to determine if the performance difference between two operators is statistically significant.

**Expected Output**: p-values and significance conclusions.

In [ ]:
# YOUR CODE HERE
# ---------------

# Compare two operators with statistical test
# operator1_results = [...]  # Multiple run results
# operator2_results = [...]  # Multiple run results

# TODO: Apply Wilcoxon test
# from scipy.stats import wilcoxon
# statistic, p_value = wilcoxon(operator1_results, operator2_results)

In [ ]:
section_divider("Summary")

## 6. Summary

### Key Takeaways

1. **Selection Pressure**: Tournament size controls exploration/exploitation balance
2. **Crossover Disruption**: Two-point generally performs well, uniform has high variance
3. **Mutation Rate**: Optimal rate often 0.01-0.05 for binary encoding
4. **Adaptive Operators**: Can improve robustness but add complexity
5. **No Universal Best**: Operator choice depends on problem characteristics
6. **Statistical Testing**: Essential for comparing operator performance

### Operator Selection Guidelines

| Problem Characteristic | Recommended Operators |
|------------------------|----------------------|
| Simple fitness landscape | Tournament (k=3), Two-point, Low mutation |
| Complex/multimodal | Tournament (k=2), Uniform, Higher mutation |
| Large search space | Rank selection, Uniform, Adaptive mutation |
| Tight constraints | Tournament (k=5), Two-point, Low mutation |
| Independent features | Any selection, Uniform crossover |
| Feature dependencies | Tournament, Two-point/HUX |

### What's Next

In **Module 2**, we'll design fitness functions:
- Time series-specific metrics
- Multi-objective optimization
- Overfitting prevention
- Constraint handling in fitness

In [ ]:
key_takeaways(["Tournament size controls exploration/exploitation balance", "Two-point generally performs well, uniform has high variance", "Optimal rate often 0.01-0.05 for binary encoding", "Can improve robustness but add complexity", "Operator choice depends on problem characteristics", "Essential for comparing operator performance"])